# Importing Libraries

In [1]:
from umap import UMAP # for UMAP latent space projections
import sys # for relatove imports of sigma

Using relative imports for sigma and its packages whilst they are being redeveloped

In [2]:
sys.path.insert(0,"..")

from sigma.utils import normalisation as norm 
from sigma.utils import visualisation as visual
from sigma.utils.load import SEMDataset
from sigma.src.utils import same_seeds
from sigma.src.dim_reduction import Experiment
from sigma.models.autoencoder import AutoEncoder
from sigma.src.segmentation import PixelSegmenter
from sigma.gui import gui
from sigma.utils.loadtem import TEMDataset

# Loading the data

In [3]:
file_path='ESC_sample.emi'
tem=TEMDataset(file_path)

ValueError: No filename matches the pattern "ESC_sample.emi"

In [ ]:
tem.nav_img.axes_manager

# For TEM datasets - define the elements of interest

In [ ]:
tem.set_xray_lines(['C_Ka', 'O_Ka', 'Mg_Ka', 'Al_Ka', 
                    'Si_Ka', 'Ti_Ka','S_Ka', 'Ca_Ka',  
                    'Fe_Ka', 'Ni_Ka'])
# dont want to bin, but we will do normalise
tem.remove_first_peak(0.1)
tem.peak_intensity_normalisation()

In [ ]:
gui.view_dataset(tem)

# Processing steps before umap projection

In [ ]:
tem.normalisation([norm.neighbour_averaging,norm.zscore])

In [ ]:
gui.view_pixel_distributions(tem,norm_list=[norm.neighbour_averaging,norm.zscore,norm.softmax],cmap='Reds')

# Latent Space Projection with UMAP

In [ ]:
data = tem.normalised_elemental_data.reshape(-1,len(tem.feature_list))
umap = UMAP(
        n_neighbors=20,
        min_dist=0.05,
        n_components=2,
        metric='euclidean'
    )
latent = umap.fit_transform(data)

# Pixel Segmentation

## Segmentation with GMM

In [ ]:
ps_gmm=PixelSegmenter(latent=latent,dataset=tem,method='GaussianMixture',method_args={'n_components' :30, 'random_state':6, 'init_params':'kmeans'} )

Initial viewing of the latent space

In [ ]:
gui.view_latent_space(ps=ps_gmm, color=True)

# Interacting with clusters in the latent space

It is possible to interact with latent space using the `interactive_latent_plot` command. 

First, demonstrate the recolouring of clusters.

To do this

1. Run the plot
2. Select clusters either with the rectangle, lasso, or by clicking them
3. Select a colour from the displayed colours at the top
4. Press the Recolour button

In [12]:
gui.interactive_latent_plot(ps=ps_gmm,ratio_to_be_shown=1.0)


This 'Recolour' functionality changes the colourmap that is used by the 'check_latent_space' function, so the recoloured clusters will appear here.

In [ ]:
gui.check_latent_space(ps=ps_gmm,show_map=True,ratio_to_be_shown=1.0)

In [14]:
gui.view_phase_map(ps=ps_gmm,alpha_cluster_map=0.6)

In [15]:
gui.view_clusters_sum_spectra(ps=ps_gmm)


SelectMultiple(options=('cluster_0', 'cluster_1', 'cluster_2', 'cluster_3', 'cluster_4', 'cluster_5', 'cluster…

Output()

In [16]:
gui.show_cluster_distribution(ps=ps_gmm)

Output()

Using NMF to understand components

# 4. Unmixing cluster spectrums using Non-negative Matrix Fatorization (NMF)

**Parameters for `gui.get_unmixed_edx_profile`**<br>
> `clusters_to_be_calculated`: *str* or *List*. If `'All'`, all cluster-spectra will be included in the data matrix for NMF. If it is a list of integers (e.g. [0,1,2,3]), only #0, #1, #2, and #3 cluster-spectra will be used as the data matrix for NMF.<br>

> `normalised`: *bool*. If `True`, the sum spectra will be normalised before NMF unmixing.<br>

> `method`: *str*. Model to be used.<br>

> `method_args`: *Dict*. Keyword arugments for the NMF method (or others). [See more detail here](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.NMF.html).<br>

In [14]:
weights, components = ps_gmm.get_unmixed_spectra_profile(clusters_to_be_calculated='All', 
                                                 n_components=7,
                                                 normalised=False, 
                                                 method='NMF', 
                                                 method_args={'init':'nndsvd'})

In [15]:
gui.show_unmixed_weights_and_compoments(ps=ps_gmm, weights=weights, components=components)

In [16]:
gui.show_abundance_map(ps=ps_gmm, weights=weights, components=components)

Output()

In [21]:
gui.show_cluster_stats(ps=ps_gmm)

Output()

Clustering with HDBSCAN

HDBSCAN will leave several points as unassigned during the segmentation. The interactive latent plots can handle this, leaving them as grey coloured in the images, and they are automaticlly excluded when clusters are selected for merging.

First perform a HDBSCAN segmetation on the dataset.

In [10]:
ps_hdb = PixelSegmenter(latent=latent, 
                    dataset=tem,
                    method="HDBSCAN",
                    method_args=dict(min_cluster_size=50, min_samples=35,
                                     max_cluster_size=int(len(latent)/10),
                                     cluster_selection_epsilon=5e-2) )

In [11]:
get_new_ps = gui.interactive_latent_plot(ps_hdb,ratio_to_be_shown=1.0)


In [ ]:
gui.check_latent_space(ps=ps_hdb,show_map=True,ratio_to_be_shown=1.0)

In [15]:
gui.view_phase_map(ps=ps_hdb,alpha_cluster_map=0.3)

In [16]:
gui.view_clusters_sum_spectra(ps=ps_hdb)

SelectMultiple(options=('cluster_0', 'cluster_1', 'cluster_2', 'cluster_4', 'cluster_5', 'cluster_6', 'cluster…

Output()

In [17]:
gui.show_cluster_distribution(ps=ps_hdb)

Output()